<a href="https://colab.research.google.com/github/ajaykumar080286/PyTorch/blob/main/19_pytorch_lstm_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [144]:
!pip install nltk

In [145]:
import torch
import nltk
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize


In [146]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""


In [147]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [148]:
tokens=word_tokenize(document.lower())

In [149]:
vocab={"<UNK>":0}

In [150]:
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)

In [151]:
len(vocab)

289

In [152]:
def text_to_indices(sentence,vocab):
  numerical_sentence = []
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab["<UNK>"])

  return numerical_sentence

In [153]:
input_sentences=document.split("\n")

In [154]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [155]:
input_numerical_sentences

[[1, 2, 3],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14, 15],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27, 28, 29, 30],
 [4, 5, 2, 31, 32, 27, 2, 6, 33],
 [2,
  31,
  32,
  27,
  2,
  6,
  5,
  34,
  35,
  30,
  36,
  2,
  31,
  6,
  7,
  37,
  38,
  39,
  34,
  40,
  28,
  41,
  12,
  42,
  30,
  15],
 [4, 5, 2, 43, 27, 2, 11, 3, 33],
 [44, 45, 46, 47, 2, 48, 49, 50],
 [51, 52],
 [51, 53, 8, 9, 10],
 [9, 54],
 [55, 8, 9, 10],
 [56, 8, 57, 58],
 [59, 60],
 [61, 59],
 [62],
 [63, 64],
 [22, 65, 66, 2, 67, 43, 68, 69, 70, 50, 71],
 [45, 72, 58, 73, 74, 46, 17, 75, 27, 76, 3, 33],
 [77, 78, 74, 73, 72, 58, 79, 80, 81, 17, 75, 27, 76, 3, 82, 83, 84, 30],
 [4, 85, 86, 87, 17, 88, 89, 33, 45, 86, 90, 17, 91, 27, 2, 89, 33],
 [92,
  93,
  94,
  95,
  80,
  96,
  78,
  36,
  97,
  85,
  22,
  87,
  17,
  89,
  22,
  65,
  98,
  99,
  73,
  100,
  2,
  91,
  30],
 [21, 65, 86, 101, 2, 102, 103, 33],
 [104,
  76,
  105,
  106,
  24,
  107,
  108,
  109,
  108,
  110,
  111,
  27,

In [156]:
print(input_numerical_sentences[:4])   # ['hello']


[[1, 2, 3], [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14, 15], [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27, 28, 29, 30], [4, 5, 2, 31, 32, 27, 2, 6, 33]]


In [157]:
training_sequence = []
for sentences in input_numerical_sentences:
  for i in range(1, len(sentences)):
    training_sequence.append(sentences[:i+1]) # ['hello']


In [158]:
len(training_sequence)

942

In [159]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [4, 5], [4, 5, 2], [4, 5, 2, 6]]

In [160]:
len_list=[]
for sentence in training_sequence:
  len_list.append(len(sentence))


In [161]:
max(len_list)

62

In [162]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [163]:
len(padded_training_sequence[941])

62

In [164]:
padded_training_sequence=torch.tensor(padded_training_sequence,dtype=torch.long)

In [165]:
padded_training_sequence.shape

torch.Size([942, 62])

In [166]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   0,   4,   5],
        ...,
        [  0,   0,   0,  ..., 285, 176, 286],
        [  0,   0,   0,  ..., 176, 286, 287],
        [  0,   0,   0,  ..., 286, 287, 288]])

In [167]:
X=padded_training_sequence[:,:-1]
y = padded_training_sequence[:,-1]

In [168]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   0,   0,   4],
        ...,
        [  0,   0,   0,  ...,   0, 285, 176],
        [  0,   0,   0,  ..., 285, 176, 286],
        [  0,   0,   0,  ..., 176, 286, 287]])

In [169]:
y

tensor([  2,   3,   5,   2,   6,   7,   8,   9,  10,  11,   3,  12,  13,  14,
         15,   6,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  18,  26,
         27,  28,  29,  30,   5,   2,  31,  32,  27,   2,   6,  33,  31,  32,
         27,   2,   6,   5,  34,  35,  30,  36,   2,  31,   6,   7,  37,  38,
         39,  34,  40,  28,  41,  12,  42,  30,  15,   5,   2,  43,  27,   2,
         11,   3,  33,  45,  46,  47,   2,  48,  49,  50,  52,  53,   8,   9,
         10,  54,   8,   9,  10,   8,  57,  58,  60,  59,  64,  65,  66,   2,
         67,  43,  68,  69,  70,  50,  71,  72,  58,  73,  74,  46,  17,  75,
         27,  76,   3,  33,  78,  74,  73,  72,  58,  79,  80,  81,  17,  75,
         27,  76,   3,  82,  83,  84,  30,  85,  86,  87,  17,  88,  89,  33,
         45,  86,  90,  17,  91,  27,   2,  89,  33,  93,  94,  95,  80,  96,
         78,  36,  97,  85,  22,  87,  17,  89,  22,  65,  98,  99,  73, 100,
          2,  91,  30,  65,  86, 101,   2, 102, 103,  33,  76, 1

In [170]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [171]:
dataset = CustomDataset(X,y)

In [172]:
len(dataset)

942

In [173]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [174]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [175]:
for input,output in dataloader:
  print(input,output)

tensor([[  0,   0,   0,  ...,   4,   5,   2],
        [  0,   0,   0,  ...,   0,  21,  65],
        [  0,   0,   0,  ...,   0,   0, 123],
        ...,
        [  0,   0,   0,  ..., 105, 106,  24],
        [  0,   0,   0,  ..., 179,   2, 180],
        [  0,   0,   0,  ...,  27,  28,  41]]) tensor([ 31,  86,  45, 194,  33,  89,  33,   2,  65, 195, 241,   2,  75,  34,
         24,  90,  24,  78,   2,   3, 166,   2, 200,   2,   5,  78, 163,  27,
        272, 107, 181,  93])
tensor([[  0,   0,   0,  ..., 240, 233, 142],
        [  0,   0,   0,  ...,   0,   0,  21],
        [  0,   0,   0,  ..., 202, 192, 127],
        ...,
        [  0,   0,   0,  ...,   0,   4,  85],
        [  0,   0,   0,  ...,  86, 136, 277],
        [  0,   0,   0,  ..., 136, 127,  17]]) tensor([151, 135,   2,  33,  27,  76, 116,  81,   2, 245,  36,  78, 262,  22,
         18,  22,  23, 142, 105,  19, 190,  78, 175, 179, 131,  30, 141,  22,
        237,  86,  22, 137])
tensor([[  0,   0,   0,  ...,  22,  23,  24],
    

In [177]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
      super().__init__()
      self.embedding=nn.Embedding(vocab_size,100)
      self.lstm=nn.LSTM(100,150,batch_first=True)
      self.fc=nn.Linear(150,vocab_size)

    def forward(self, x):
      embedded = self.embedding(x)
      intermediate_hidden_states, (final_hidden_state, final_cell_state)=self.lstm(embedded)
      output = self.fc(final_hidden_state.squeeze(0))
      return output


In [179]:
x=nn.Embedding(289,100)
y=nn.LSTM(100,150,batch_first=True)

In [185]:
a=dataset[0][0].unsqueeze(0)

In [188]:
b=x(a)

In [191]:
c,d=y(b)

In [195]:
c.shape  #set of intermidate hidden state

torch.Size([1, 61, 150])

In [196]:
e,f=d

In [197]:
e.shape

torch.Size([1, 1, 150])

In [198]:
f.shape

torch.Size([1, 1, 150])

In [199]:
z=nn.Linear(150,289)

In [202]:
z(f).squeeze(0).shape

torch.Size([1, 289])

In [203]:
model = LSTMModel(len(vocab))

In [204]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [205]:
model.to(device)

LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [206]:
epochs = 2
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [207]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    optimizer.zero_grad()
    output = model(batch_x)
    loss = criterion(output, batch_y)
    loss.backward()
    optimizer.step()
    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 166.1698
Epoch: 2, Loss: 145.8168


In [208]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())
  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)
  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)
  # send to model
  output = model(padded_text)
  # predicted index
  value, index = torch.max(output, dim=1)
  # merge with text
  return text + " " + list(vocab.keys())[index]



In [209]:
prediction(model, vocab, "The course follows a monthly")

'The course follows a monthly the'

In [210]:
input_text = "hi how are"
for i in range(10):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text


hi how are the
hi how are the the
hi how are the the ?
hi how are the the ? ?
hi how are the the ? ? the
hi how are the the ? ? the ?
hi how are the the ? ? the ? ?
hi how are the the ? ? the ? ? the
hi how are the the ? ? the ? ? the ?
hi how are the the ? ? the ? ? the ? ?
